# Load and Initial Cleaning — AI Jobs 2026
This notebook loads ai_jobs_global_2026.csv, performs basic cleaning (null checks, filling missing salary with median, marking missing required_skills), and shows distributions for experience_level and country.

In [ ]:
# Imports
import pandas as pd
import numpy as np
pd.options.display.max_columns = 200
pd.options.display.width = 120

In [ ]:
# Load dataset
csv_path = 'ai_jobs_global_2026.csv'
with open(csv_path, 'r', encoding='utf-8', errors='replace') as f:
    df = pd.read_csv(f)
print('Loaded:', csv_path, '->', df.shape)
df.head()

In [ ]:
# Quick inspection: head, info, describe
print('--- head() ---')
display(df.head())
print('\n--- info() ---')
df.info()
print('\n--- describe(include=\'all\') ---')
display(df.describe(include='all'))

In [ ]:
# Check for null values per column
missing = df.isnull().sum()
print('Columns with missing values:')
print(missing[missing > 0])

In [ ]:
# Handle missing salary data robustly using salary_min and salary_max
# Create a unified 'salary' column: mean of min/max when available, else use whichever exists
if 'salary' not in df.columns:
    if 'salary_min' in df.columns or 'salary_max' in df.columns:
        # coerce to numeric
        if 'salary_min' in df.columns:
            df['salary_min'] = pd.to_numeric(df['salary_min'], errors='coerce')
        if 'salary_max' in df.columns:
            df['salary_max'] = pd.to_numeric(df['salary_max'], errors='coerce')
        if 'salary_min' in df.columns and 'salary_max' in df.columns:
            df['salary'] = df[['salary_min','salary_max']].mean(axis=1)
        elif 'salary_min' in df.columns:
            df['salary'] = df['salary_min']
        else:
            df['salary'] = df['salary_max']
        print('Created unified column "salary" from salary_min/salary_max')
    else:
        print('No salary_min/salary_max columns found; no salary column created')
# Now fill missing values in 'salary' with its median if present
if 'salary' in df.columns:
    med = pd.to_numeric(df['salary'], errors='coerce').median()
    df['salary'] = pd.to_numeric(df['salary'], errors='coerce').fillna(med)
    print('Filled missing values in salary with median =', med)
    print('Remaining nulls in salary:', df['salary'].isnull().sum())
else:
    print('No salary column to fill.')

In [ ]:
# Specifically handle nulls in 'required_skills' by marking them
if 'required_skills' in df.columns:
    before = df['required_skills'].isnull().sum()
    df['required_skills'] = df['required_skills'].fillna('To be extracted')
    after = df['required_skills'].isnull().sum()
    print("'required_skills': filled {} missing values; remaining nulls = {}".format(before - after, after))
else:
    print("Column 'required_skills' not found in the dataset.")

In [ ]:
# Show distributions (value_counts) for 'experience_level' and 'country'
for col in ['experience_level', 'country']:
    if col in df.columns:
        print('\nValue counts for', col)
        print(df[col].value_counts(dropna=False))
    else:
        print(f"Column '{col}' not found.")

In [ ]:
# Text preprocessing for 'job_description' using NLTK
import re
from tqdm.notebook import tqdm
import nltk
# Ensure necessary NLTK data (download if missing)
for pkg in ['punkt','wordnet','omw-1.4','stopwords','averaged_perceptron_tagger']:
    try:
        nltk.data.find(pkg if pkg!='punkt' else 'tokenizers/punkt')
    except LookupError:
        nltk.download(pkg)
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def nltk_pos_to_wordnet(nltk_pos):
    if nltk_pos.startswith('J'):
        return wordnet.ADJ
    elif nltk_pos.startswith('V'):
        return wordnet.VERB
    elif nltk_pos.startswith('N'):
        return wordnet.NOUN
    elif nltk_pos.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def preprocess_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # Use regex tokenizer to avoid punkt-related lookup issues
    tokens = re.findall(r'\b[a-z]+\b', text)
    pos_tags = nltk.pos_tag(tokens)
    cleaned = []
    for token, pos in pos_tags:
        if token in stop_words or not token.isalpha():
            continue
        wn_pos = nltk_pos_to_wordnet(pos)
        lemma = lemmatizer.lemmatize(token, wn_pos)
        cleaned.append(lemma)
    return ' '.join(cleaned)

if 'job_description' in df.columns:
    print('Preprocessing job_description...')
    # Use tqdm over the Series to avoid pandas apply pickling issues
    descriptions = df['job_description'].fillna('').astype(str).tolist()
    cleaned = []
    for txt in tqdm(descriptions):
        cleaned.append(preprocess_text(txt))
    df['clean_description'] = cleaned
    print('Sample cleaned description:')
    display(df[['job_description','clean_description']].head())
else:
    print("Column 'job_description' not found in dataset.")

In [ ]:
# Verification: show salary and cleaned description summaries
print('Data shape:', df.shape)
if 'salary' in df.columns:
    print('\nSalary summary:')
    display(df['salary'].describe())
    print('Nulls in salary:', df['salary'].isnull().sum())
else:
    print('No unified salary column present.')

if 'clean_description' in df.columns:
    print('\nSample cleaned descriptions:')
    display(df[['clean_description']].head())
    print('Count of empty cleaned descriptions:', (df['clean_description']=='').sum())
else:
    print('No clean_description column present.')

print('\nTop experience_level counts:')
if 'experience_level' in df.columns:
    display(df['experience_level'].value_counts().head(20))
else:
    print('No experience_level column')

print('\nTop country counts:')
if 'country' in df.columns:
    display(df['country'].value_counts().head(20))
else:
    print('No country column')

In [ ]:
# Save cleaned dataset to CSV
out_path = 'ai_jobs_global_2026_cleaned.csv'
# Only save if we have the expected cleaned columns
if 'clean_description' in df.columns or 'salary' in df.columns:
    df.to_csv(out_path, index=False)
    print('Saved cleaned dataset to', out_path)
else:
    print('Dataframe missing expected cleaned columns; not saving.')